# Handwritten Digit Recognition — CNN + Transformer Hybrid

Based on: *"Handwriting Recognition through Neural Networks: Enhancing Accuracy and Performance"*  
S. Suman Rajest, R. Regin — CAJMNS, 2024

**Architecture**: CNN feature extractor → Transformer encoder → classification head  
**Device**: Apple Metal (MPS) GPU acceleration

## 1. Imports & Setup

In [ ]:
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset

# ── Device ───────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print(f"Using MPS (Apple Metal): {DEVICE}")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using CUDA: {DEVICE}")
else:
    DEVICE = torch.device("cpu")
    print(f"Using CPU: {DEVICE}")

# ── Reproducibility ──────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Config ───────────────────────────────────────────
BATCH_SIZE = 256
NUM_WORKERS = 4
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

## 2. Load Dataset from HuggingFace

Loads the preprocessed dataset from [leobottaro/handwritten-digits-pack](https://huggingface.co/datasets/leobottaro/handwritten-digits-pack).

In [ ]:
ds = load_dataset("leobottaro/handwritten-digits-pack")
print(f"Splits: {list(ds.keys())}")

# Convert HF dataset to torch tensors for DataLoader
# Note: Parquet stores images as PNG bytes, decoded as uint8 [0..255]
def hf_to_tensor(example):
    img = np.array(example["image"], dtype=np.float32) / 255.0  # [0..255] → [0..1]
    return {
        "image": torch.from_numpy(img).unsqueeze(0),  # (1, 28, 28)
        "label": example["label"],
    }

ds = ds.map(hf_to_tensor)
ds.set_format(type="torch", columns=["image", "label"])

# Extract splits
train_ds = ds["train"]
val_ds   = ds["validation"]

NUM_CLASSES = 10
print(f"Train: {len(train_ds):,} samples")
print(f"Val:   {len(val_ds):,} samples")
print(f"Classes: {NUM_CLASSES}")
print(f"Image shape: {train_ds[0]['image'].shape}, dtype: {train_ds[0]['image'].dtype}")
print(f"Value range: [{train_ds[0]['image'].min():.4f}, {train_ds[0]['image'].max():.4f}]")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 3. Model — CNN + Transformer Hybrid (per paper)

**Paper architecture**:  
- **CNN**: Low-level feature extraction — edges, shapes, character details  
  Multiple conv+pool layers refine features before passing to transformer  
- **Transformer**: Self-attention captures long-range spatial dependencies  
  across the feature map, handling complex handwriting patterns  
- **Classification head**: FC layers map transformer output → 10 digit classes

In [ ]:
class PositionalEncoding(nn.Module):
    """Learned positional encoding for spatial feature sequence."""
    def __init__(self, seq_len: int, d_model: int):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)

    def forward(self, x):
        # x: (B, seq_len, d_model)
        return x + self.pos_embed


class CNNTransformer(nn.Module):
    """CNN feature extractor + Transformer encoder for digit classification.

    As described in Rajest & Regin (2024):
    - CNN component: low-level feature extraction (edges, shapes)
    - Transformer component: spatial dependencies via self-attention
    """

    def __init__(self, num_classes: int = 10, d_model: int = 256,
                 nhead: int = 8, num_layers: int = 4,
                 dim_feedforward: int = 512, dropout: float = 0.1):
        super().__init__()

        # ── CNN Feature Extractor ──
        # Input: (B, 1, 28, 28)
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True), nn.MaxPool2d(2),  # → (32, 14, 14)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True), nn.MaxPool2d(2),  # → (64, 7, 7)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),                  # → (128, 7, 7)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, d_model, 3, padding=1), nn.BatchNorm2d(d_model),
            nn.ReLU(inplace=True), nn.MaxPool2d(2), # → (256, 3, 3)
        )

        # ── Spatial → Sequence ──
        self.seq_len = 3 * 3  # 9 spatial positions
        self.pos_encoder = PositionalEncoding(self.seq_len, d_model)
        self.dropout_embed = nn.Dropout(dropout)

        # ── Transformer Encoder ──
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # ── Classification Head ──
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # CNN feature extraction
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)  # (B, d_model, 3, 3)

        # Flatten spatial dims → sequence: (B, d_model, H, W) → (B, H*W, d_model)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)  # (B, H*W, C) = (B, 9, d_model)

        # Positional encoding + dropout
        x = self.pos_encoder(x)
        x = self.dropout_embed(x)

        # Transformer encoder
        x = self.transformer(x)  # (B, 9, d_model)

        # Mean pool over sequence → classification
        x = x.mean(dim=1)  # (B, d_model)
        x = self.classifier(x)
        return x


# ── Instantiate ──────────────────────────────────────
model = CNNTransformer(num_classes=NUM_CLASSES).to(DEVICE)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"CNNTransformer: {total:,} params ({trainable:,} trainable)")

# Quick shape test
dummy = torch.randn(4, 1, 28, 28).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f"Input: {list(dummy.shape)} → Output: {list(out.shape)}")

## 4. Training Setup

Per paper: **supervised learning**, **categorical cross-entropy loss**, **cross-validation** (implemented as hold-out validation on large dataset), metrics: accuracy / precision / recall / F1.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
)

# For mixed-precision on MPS (helps on newer chips)
use_amp = DEVICE.type == "mps"
scaler = torch.amp.GradScaler("mps", enabled=use_amp) if use_amp else None

print(f"Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"Scheduler: ReduceLROnPlateau (patience=5, factor=0.5)")
print(f"AMP: {'enabled' if use_amp else 'disabled'}")

## 5. Training Loop

Includes per-epoch validation, early stopping, and checkpointing of best model.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch:2d} [train]", leave=False)
    for X, y in pbar:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        if scaler is not None:
            with torch.amp.autocast("mps"):
                logits = model(X)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += X.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.3f}")

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds, all_labels = [], []

    for X, y in tqdm(loader, desc="Validating", leave=False):
        X, y = X.to(DEVICE), y.to(DEVICE)
        logits = model(X)
        loss = criterion(logits, y)
        total_loss += loss.item() * X.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += X.size(0)
        all_preds.append(logits.argmax(1).cpu())
        all_labels.append(y.cpu())

    preds = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    return total_loss / total, correct / total, preds, labels

In [ ]:
# ── Training state ───────────────────────────────────
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_epoch = 0
patience_counter = 0
EARLY_STOP_PATIENCE = 12
CHECKPOINT_PATH = "model_best.pt"
start_epoch = 1

# ── Resume from checkpoint if exists ────────────────
if Path(CHECKPOINT_PATH).exists():
    print(f"Found checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    best_val_acc = ckpt["val_acc"]
    best_epoch = ckpt["epoch"]
    start_epoch = ckpt["epoch"] + 1
    if "history" in ckpt:
        history = ckpt["history"]
    # Re-create scheduler with loaded optimizer state
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
    )
    print(f"Resuming from epoch {start_epoch} (best val_acc={best_val_acc:.4f} at epoch {best_epoch})")
else:
    print("No checkpoint found — starting fresh")

print(f"Training epochs {start_epoch}–{NUM_EPOCHS}, early stop patience={EARLY_STOP_PATIENCE}")
print(f"{'='*60}")

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, epoch)
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:2d} | train loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"val loss={val_loss:.4f} acc={val_acc:.4f}", end="")

    # Checkpoint best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc": val_acc,
            "history": history,
        }, CHECKPOINT_PATH)
        print("  ✓ best", end="")
    else:
        patience_counter += 1
        print(f"  ({patience_counter}/{EARLY_STOP_PATIENCE})", end="")

    print(f"  lr={scheduler.get_last_lr()[0]:.2e}")

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\n{'='*60}")
print(f"Best val acc: {best_val_acc:.4f} (epoch {best_epoch})")

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs, history["train_loss"], "b-", label="Train")
ax1.plot(epochs, history["val_loss"], "r-", label="Val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Cross-Entropy Loss")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history["train_acc"], "b-", label="Train")
ax2.plot(epochs, history["val_acc"], "r-", label="Val")
ax2.axhline(y=best_val_acc, color="g", linestyle="--", alpha=0.5, label=f"Best ({best_val_acc:.4f})")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_title("Classification Accuracy")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluation — Accuracy, Precision, Recall, F1

Per paper: evaluate on held-out test set using all four metrics.

In [ ]:
# Load best checkpoint and evaluate
ckpt = torch.load(CHECKPOINT_PATH, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")

val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

acc  = accuracy_score(val_labels, val_preds)
prec = precision_score(val_labels, val_preds, average="macro")
rec  = recall_score(val_labels, val_preds, average="macro")
f1   = f1_score(val_labels, val_preds, average="macro")

print(f"\n{'='*40}")
print(f"Test Accuracy:  {acc:.4f}")
print(f"Test Precision: {prec:.4f} (macro)")
print(f"Test Recall:    {rec:.4f} (macro)")
print(f"Test F1-Score:  {f1:.4f} (macro)")
print(f"{'='*40}")

# Per-class metrics
per_class_prec = precision_score(val_labels, val_preds, average=None)
per_class_rec  = recall_score(val_labels, val_preds, average=None)
per_class_f1   = f1_score(val_labels, val_preds, average=None)

print(f"\n{'Digit':>6} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"{'-'*40}")
for d in range(10):
    print(f"{d:>6} {per_class_prec[d]:>10.4f} {per_class_rec[d]:>10.4f} {per_class_f1[d]:>10.4f}")

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(val_labels, val_preds)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix (acc={acc:.4f})")
ax.set_xticks(range(10)); ax.set_yticks(range(10))

# Annotate
for i in range(10):
    for j in range(10):
        text = f"{cm[i,j]}" if cm[i,j] < cm.max() * 0.7 else f"{cm[i,j]}"
        color = "white" if cm[i,j] > cm.max() * 0.5 else "black"
        ax.text(j, i, text, ha="center", va="center", fontsize=8, color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 9. Sample Predictions

In [ ]:
@torch.no_grad()
def show_predictions(loader, n_cols=10, n_rows=3):
    model.eval()
    X_batch, y_batch = next(iter(loader))
    X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
    logits = model(X_batch)
    preds = logits.argmax(1)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.4, n_rows * 1.5))
    for i, ax in enumerate(axes.flat):
        if i >= len(X_batch): break
        img = X_batch[i, 0].cpu()
        true_lbl = y_batch[i].item()
        pred_lbl = preds[i].item()
        ax.imshow(img, cmap="gray_r")
        color = "green" if true_lbl == pred_lbl else "red"
        ax.set_title(f"T:{true_lbl} P:{pred_lbl}", color=color, fontsize=9, fontweight="bold")
        ax.axis("off")
    plt.suptitle("Predictions — Green=correct, Red=wrong", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

show_predictions(val_loader)

## 10. Wrong Predictions

Samples the model got wrong — shows true label vs predicted label.

In [ ]:
@torch.no_grad()
def show_wrong_predictions(val_dataset, model, max_wrong=30):
    """Find misclassified samples and display them in a grid."""
    model.eval()
    wrong_X, wrong_true, wrong_pred = [], [], []

    loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        logits = model(X_batch)
        preds = logits.argmax(1)
        mask = preds != y_batch
        if mask.any():
            wrong_X.append(X_batch[mask].cpu())
            wrong_true.append(y_batch[mask].cpu())
            wrong_pred.append(preds[mask].cpu())
        if sum(len(w) for w in wrong_X) >= max_wrong:
            break

    wrong_X = torch.cat(wrong_X)[:max_wrong]
    wrong_true = torch.cat(wrong_true)[:max_wrong]
    wrong_pred = torch.cat(wrong_pred)[:max_wrong]

    n = len(wrong_X)
    if n == 0:
        print("No wrong predictions found!")
        return

    cols = min(10, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.6))
    if rows == 1:
        axes = axes[np.newaxis, :]

    for i in range(rows * cols):
        r, c = divmod(i, cols)
        ax = axes[r, c]
        if i < n:
            img = wrong_X[i, 0]
            ax.imshow(img, cmap="gray_r")
            ax.set_title(f"T:{wrong_true[i].item()}→P:{wrong_pred[i].item()}",
                        color="red", fontsize=8, fontweight="bold")
        ax.axis("off")

    fig.suptitle(f"Wrong Predictions ({n} of {len(val_dataset)} samples)",
                 fontsize=13, fontweight="bold", color="red")
    plt.tight_layout()
    plt.show()

show_wrong_predictions(val_ds, model)

## 11. CNN Layer Visualizations

What each CNN block "sees" — feature map activations for a sample digit through every conv block.

In [ ]:
@torch.no_grad()
def visualize_cnn_layers(model, sample_input, layer_names=None):
    """Show feature maps from each CNN block for a single input image.

    Hooks into conv1, conv2, conv3, conv4 to capture activations,
    then displays first 16 channels of each as a grid.
    """
    model.eval()
    activations = {}

    def hook_fn(name):
        def fn(module, inp, out):
            activations[name] = out.detach().cpu()
        return fn

    hooks = []
    for name in ["conv1", "conv2", "conv3", "conv4"]:
        module = getattr(model, name)
        hooks.append(module.register_forward_hook(hook_fn(name)))

    # Forward pass
    sample = sample_input.unsqueeze(0).to(DEVICE)  # (1, 1, 28, 28)
    model(sample)

    for h in hooks:
        h.remove()

    # Plot
    fig, axes = plt.subplots(4, 16, figsize=(20, 6))
    for row, name in enumerate(["conv1", "conv2", "conv3", "conv4"]):
        feat = activations[name][0]  # (C, H, W)
        n_channels = feat.shape[0]
        for col in range(16):
            ax = axes[row, col]
            if col < n_channels:
                ax.imshow(feat[col], cmap="viridis")
            ax.axis("off")
        axes[row, 0].set_ylabel(f"{name}\n{list(feat.shape)}",
                                fontsize=8, fontweight="bold", rotation=0,
                                labelpad=40, ha="right", va="center")

    fig.suptitle("CNN Feature Maps — conv1 → conv2 → conv3 → conv4\n"
                 "(showing first 16 channels per block)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Pick a sample digit from validation set
sample = val_ds[0]
sample_img = sample["image"]  # (1, 28, 28)
sample_label = sample["label"]
print(f"Visualizing layers for digit: {sample_label}")
visualize_cnn_layers(model, sample_img)

## 12. ONNX Export

Export trained model to ONNX format for platform-independent inference (CoreML, TensorRT, OpenVINO, etc.).

In [ ]:
import onnx
import onnxruntime as ort

# Load best checkpoint onto CPU for export (ONNX export needs CPU tracing)
ckpt = torch.load(CHECKPOINT_PATH, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
model.cpu()  # ONNX tracing on CPU
print(f"Loaded checkpoint epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")

ONNX_PATH = "model.onnx"
dummy_input = torch.randn(1, 1, 28, 28)  # (B, C, H, W)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
)

# Verify exported model
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

# File size
size_mb = Path(ONNX_PATH).stat().st_size / (1024 * 1024)
print(f"ONNX model saved: {ONNX_PATH} ({size_mb:.1f} MB)")
print(f"ONNX opset: {onnx_model.opset_import[0].version}")
print("ONNX check: PASSED")

# Move model back to device for further use
model.to(DEVICE)

## 13. ONNX Inference

Run inference with ONNX Runtime. Compare outputs against PyTorch model — they should match exactly.

In [ ]:
# Create ONNX Runtime session (CPU)
ort_session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

# Run inference on a few validation samples
sample_X = torch.stack([val_ds[i]["image"] for i in range(10)]).numpy()
sample_y = torch.tensor([val_ds[i]["label"] for i in range(10)]).numpy()

# ONNX inference
onnx_outputs = ort_session.run(None, {"input": sample_X})
onnx_logits = onnx_outputs[0]  # (N, 10)
onnx_preds = onnx_logits.argmax(axis=1)

# PyTorch inference for comparison
with torch.no_grad():
    model.eval()
    torch_logits = model(torch.from_numpy(sample_X).to(DEVICE)).cpu().numpy()
    torch_preds = torch_logits.argmax(axis=1)

# Compare
max_diff = np.abs(onnx_logits - torch_logits).max()
pred_match = (onnx_preds == torch_preds).all()

print(f"Max logit difference (ONNX vs PyTorch): {max_diff:.2e}")
print(f"Predictions match: {pred_match}")
print(f"\n{'Idx':>4} {'True':>5} {'Torch':>5} {'ONNX':>5} {'Match':>6}")
print(f"{'-'*32}")
for i in range(10):
    match = "✓" if torch_preds[i] == onnx_preds[i] else "✗"
    print(f"{i:>4} {sample_y[i]:>5} {torch_preds[i]:>5} {onnx_preds[i]:>5} {match:>6}")

print(f"\nONNX inference ready: {ONNX_PATH}")
print(f"Usage: ort_session.run(None, {{'input': image_array}}) → logits")